In [1]:
%reload_ext autoreload
%autoreload 2
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import os

print("gseapy version:", gp.__version__)

ModuleNotFoundError: No module named 'gseapy'

In [2]:
## define path
basedir = "/Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx"

sub_out = "chaphama_260206_MSigDB_Hallmark_2020_stat_blastx_filtered/"  # edited on 260209
out_folder_path = f"{basedir}/Users/kyokokurihara/Lab/projects/2507blastx/output/251225GSEA/"
out_path = out_folder_path + sub_out

input_folder_path = "/Users/kyokokurihara/Lab/projects/2507blastx/data/251215_rna_seq/To_kurihara_downloaded260209_chaphama/output"  # 260209
# make directories
if not(os.path.exists(out_folder_path)):
    os.mkdir(out_folder_path)
if not(os.path.exists(out_path)):
    os.mkdir(out_path)
print("saving files to:", out_path)

saving files to: /Users/kyokokurihara/Lab/projects/2507blastx/output/251225GSEA/chaphama_260206_MSigDB_Hallmark_2020_stat_blastx_filtered/


In [3]:
def plot_prerank(df, outdir_path, metric, gene_col, gene_sets):
    """
    prerank tool plot.

    metric: ranking metric
    gene_col: gene name col for enrichr
    """
    # cols: gene_name_hs, gene_name_gallus, baseMean, log2FoldChange, lfcSE, stat, pvalue, padj    
    df2 = df.copy()
    
    # drop NaN
    df2 = df2.dropna(subset=[gene_col, metric])
    df2[gene_col] = df2[gene_col].astype(str)
    # df2[gene_col] = df2[gene_col].astype(str).str.upper()
    
    # for duplicate genes, keep the one with the larger absolute value 
    # (for ties: 1. pvalue, 2. gene name)
    df2["_abs"] = df2[metric].astype(float).abs()
    df2["_p"] = df2["pvalue"].astype(float).fillna(1.0)

    df2 = (df2.sort_values([gene_col, "_abs", "_p", gene_col],
                           ascending=[True, False, True, True],
                           kind="mergesort").drop_duplicates(subset=[gene_col], keep="first"))

    # ranking (for ties: 1. pvalue, 2. gene name)
    df2 = df2.sort_values([metric, "_p", gene_col],
                          ascending=[False, True, True],
                          kind="mergesort")
    rnk = df2[[gene_col, metric]]
    print("\nrnk\n", rnk.head(3))

    # prerank
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=gene_sets,
        outdir=f"{outdir_path}_{gene_sets}_{metric}_{gene_col}",
        seed=0,
        threads=4,
        verbose=True
    )

    # plot top 5 pathways
    terms = pre_res.res2d["Term"]
    pre_res.plot(
        terms=terms[:5], 
        legend_kws={'loc': (1.15, 0)},
        ofname=f"{outdir_path}_{gene_sets}_{metric}_{gene_col}_top5_enriched.png"
    )

In [4]:
# check Enricher library
names = gp.get_library_name()
# print("all library:", names)
print("KEGG library:", [k for k in names if k.startswith("KEGG_")])
print("MSigDB library:", [m for m in names if m.startswith("MSigDB_")])
print("Reactome library:", [r for r in names if r.startswith("Reactome")])
print("GO library:", [g for g in names if g.startswith("GO_Biological")])

KEGG library: ['KEGG_2013', 'KEGG_2015', 'KEGG_2016', 'KEGG_2019_Human', 'KEGG_2019_Mouse', 'KEGG_2021_Human', 'KEGG_2026']
MSigDB library: ['MSigDB_Computational', 'MSigDB_Hallmark_2020', 'MSigDB_Oncogenic_Signatures']
Reactome library: ['Reactome_2022', 'Reactome_Pathways_2024']
GO library: ['GO_Biological_Process_2021', 'GO_Biological_Process_2023', 'GO_Biological_Process_2025']


In [5]:
configs = {
    "metric": "stat",
    "gene_col": "gene_name_hs",
    "gene_sets": "MSigDB_Hallmark_2020" 

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if not(fname.endswith("_hs_gallus_and_Gene.txt")):
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        project = pd.read_table(path)

        # plot prerank
        proname = fname.split("_")[1]
        print("\nprocessing...", proname)
        plot_prerank(project, out_path + proname, configs["metric"], configs["gene_col"], configs["gene_sets"])

2026-02-09 13:57:38,548 [WARNING] Duplicated values found in preranked stats: 1.08% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-02-09 13:57:38,548 [INFO] Parsing data files for GSEA.............................
2026-02-09 13:57:38,556 [INFO] Enrichr library gene sets already downloaded in: /Users/kyokokurihara/.cache/gseapy, use local file
2026-02-09 13:57:38,562 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-02-09 13:57:38,563 [INFO] 0050 gene_sets used for further statistical testing.....
2026-02-09 13:57:38,563 [INFO] Start to run GSEA...Might take a while..................



processing... PRJNA577590

rnk
       gene_name_hs      stat
11736       ATRNL1  4.864696
15087         RRAD  3.511008
5100        APCDD1  3.080048


2026-02-09 13:57:43,430 [INFO] Congratulations. GSEApy runs successfully................

2026-02-09 13:57:43,718 [WARNING] Duplicated values found in preranked stats: 1.60% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-02-09 13:57:43,718 [INFO] Parsing data files for GSEA.............................
2026-02-09 13:57:43,726 [INFO] Enrichr library gene sets already downloaded in: /Users/kyokokurihara/.cache/gseapy, use local file
2026-02-09 13:57:43,731 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-02-09 13:57:43,731 [INFO] 0050 gene_sets used for further statistical testing.....
2026-02-09 13:57:43,732 [INFO] Start to run GSEA...Might take a while..................



processing... PRJNA622813

rnk
       gene_name_hs       stat
8499         DDX60  32.073228
19286        ZNFX1  27.803089
10787      TNFAIP2  20.985157


2026-02-09 13:57:49,036 [INFO] Congratulations. GSEApy runs successfully................

2026-02-09 13:57:49,292 [WARNING] Duplicated values found in preranked stats: 1.29% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-02-09 13:57:49,292 [INFO] Parsing data files for GSEA.............................
2026-02-09 13:57:49,302 [INFO] Enrichr library gene sets already downloaded in: /Users/kyokokurihara/.cache/gseapy, use local file
2026-02-09 13:57:49,306 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-02-09 13:57:49,307 [INFO] 0050 gene_sets used for further statistical testing.....
2026-02-09 13:57:49,307 [INFO] Start to run GSEA...Might take a while..................



processing... PRJNA612882

rnk
       gene_name_hs      stat
11179          RET  4.157465
12398         SPEG  4.062659
20505         ESAM  3.823232


2026-02-09 13:57:54,798 [INFO] Congratulations. GSEApy runs successfully................

